# Point-pattern KDE: how bandwidth changes the intensity surface

This lab turns a synthetic point pattern into a continuous Gaussian kernel intensity surface. It is descriptive, not a statistical test of clustering.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt
ROOT = Path.cwd()
if not (ROOT / 'implementations').exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'implementations/python/point-pattern-kde'))
from point_pattern_kde import gaussian_kde


## 1. Create a reproducible point pattern

Two synthetic event concentrations are placed inside a rectangular study window. The coordinates are planar teaching coordinates, not a real geographic dataset.

In [ ]:
rng = np.random.default_rng(17)
left = rng.normal(loc=(3, 3), scale=(0.65, 0.65), size=(45, 2))
right = rng.normal(loc=(7, 7), scale=(0.8, 0.8), size=(45, 2))
points = np.vstack([left, right])
points = points[((points[:, 0] >= 0) & (points[:, 0] <= 10) & (points[:, 1] >= 0) & (points[:, 1] <= 10))]
fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(points[:, 0], points[:, 1], alpha=.7)
ax.set(xlim=(0, 10), ylim=(0, 10), title=f'Synthetic point pattern ({len(points)} events)', xlabel='x', ylabel='y')
ax.set_aspect('equal'); ax.grid(True); plt.show()


## 2. Compare bandwidths

The Gaussian kernel is fixed; only the bandwidth changes. A small bandwidth retains narrow peaks. A large bandwidth spreads each event's contribution farther across the window.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharex=True, sharey=True)
for ax, bandwidth in zip(axes, [0.35, 0.75, 1.5]):
    density, xs, ys = gaussian_kde(points, bounds=(0, 10, 0, 10), grid_shape=(90, 90), bandwidth=bandwidth)
    contour = ax.contourf(xs, ys, density, levels=16, cmap='viridis')
    ax.scatter(points[:, 0], points[:, 1], s=5, c='white', alpha=.5)
    ax.set_title(f'Bandwidth = {bandwidth}'); ax.set_aspect('equal')
    fig.colorbar(contour, ax=ax, shrink=.8, label='Intensity per map-unit²')
plt.tight_layout(); plt.show()


## 3. Add event weights

Weights change the contribution of each event. The resulting intensity is no longer simply event count per area; its units follow the weights.

In [ ]:
weights = np.ones(len(points))
weights[:10] = 3
weighted_density, xs, ys = gaussian_kde(points, bounds=(0, 10, 0, 10), grid_shape=(90, 90), bandwidth=.75, weights=weights)
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(weighted_density, extent=(0, 10, 0, 10), origin='lower', cmap='magma')
ax.scatter(points[:, 0], points[:, 1], s=8, c='cyan', alpha=.6)
ax.set_title('Weighted event intensity'); ax.set_aspect('equal'); fig.colorbar(im, ax=ax, label='Weighted intensity'); plt.show()


## Investigate

1. Which bandwidth preserves the two separate concentrations?
2. Which bandwidth begins to merge them?
3. Move a cluster close to the boundary. Why can the estimate be biased downward there?
4. Does a finer grid create more information, or does it sample the same estimated surface more densely?
5. Why is this KDE surface not a test of statistically significant clustering?